In [ ]:
#Install
# !pip install torch==2.8.0
# !pip install transformers==4.55.4
# !pip install datasets==3.6.0
# !pip install bitsandbytes==0.47.0
# !pip install unsloth==2025.9.7
# !pip install trl==0.22.2
# !pip install streamlit==1.50.0
# !pip install pyngrok==7.4.0

## Reference Links

* [https://docs.unsloth.ai/get-started/fine-tuning-llms-guide](https://docs.unsloth.ai/get-started/fine-tuning-llms-guide)
* [https://huggingface.co/docs/peft/main/en/conceptual_guides/lora](https://huggingface.co/docs/peft/main/en/conceptual_guides/lora)
* [https://huggingface.co/blog/4bit-transformers-bitsandbytes](https://huggingface.co/blog/4bit-transformers-bitsandbytes)

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit"
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

train_prompt = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Answer the following medical question with clarity, accuracy, and patient safety in mind.
First, carefully analyze the problem step by step to ensure logical reasoning, and then provide a clear and concise final response.
If relevant, highlight diagnostic considerations, treatment options, and red-flag safety notes.
Do not include irrelevant details or speculative information. Please answer the following medical question.


### Question:
{}

### Chain of Thought:
<think>
{}
</think>

### Final Response:
{}
"""


EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    inputs = examples["Question"]
    complex_cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []
    for question, cot, response in zip(inputs, complex_cots, outputs):
        # Append the EOS token to the response if it's not already there
        if not response.endswith(tokenizer.eos_token):
            response += tokenizer.eos_token
        text = train_prompt.format(question, cot, response)
        texts.append(text)
    return {"text": texts}


from datasets import load_dataset

dataset = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    "en",
    split="train",
    trust_remote_code=True,
)


#Format it
dataset = dataset.map(formatting_prompts_func, batched=True)


In [ ]:
question = dataset[10]['Question']
question

'In a patient with dermatomyositis as indicated by fatigue, muscle weakness, a scaly rash, elevated creatine kinase-MB, anti-Jo-1 antibodies, and perimysial inflammation, which type of cancer is most often associated with this condition?'

In [ ]:
response = dataset[10]['Response']
response

'In patients with dermatomyositis, ovarian cancer is the most often associated malignancy, particularly in women. While other types of cancer such as breast, lung, and gastric can also be associated with dermatomyositis, ovarian cancer has a relatively strong and specific link, especially in the context of developing this autoimmune condition. It’s important for individuals with dermatomyositis to undergo appropriate cancer screening for early detection and management, particularly for ovarian cancer.'

In [ ]:
# Another question

sample_question = """A 22-year-old woman presented with right lower quadrant and suprapubic abdominal pain.
A CT scan of the abdomen and pelvis revealed a thickened tubular structure in the right lower quadrant with a hypodense mass directly adjacent to it. Radiographic interpretation included an adnexal mass, but appendicitis could not be ruled out. The patient underwent emergent laparoscopy.
During surgery, a large ovary, thickened fallopian tube, and adnexal fluid were visualized on the right side.
The appendix appeared normal.
A significant number of adhesions that involved the liver were noted. Whats the diagnosis?"""


Correct_Output = """The diagnosis was Fitz-Hugh-Curtis syndrome."""



In [ ]:
inference_prompt = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Below is a medical question. Answer the following medical question with clarity, accuracy, and patient safety in mind.
First, carefully analyze the problem step by step to ensure logical reasoning, and then provide a clear and concise final response.
If relevant, highlight diagnostic considerations, treatment options, and red-flag safety notes.
Do not include irrelevant details or speculative information. Please answer the following medical question.


### Question:
{}

### Final Response:

"""

In [ ]:


inputs = tokenizer(
    [inference_prompt.format(question) + tokenizer.eos_token],
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=5000,
    eos_token_id=tokenizer.eos_token_id,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print(response[0].split("### Final Response:")[1])




://www.ninds.nih.gov/health-information/brain-and-spinal-cord-tumors-adults/diagnosis-treatment-and-supportive-care

Dermatomyositis is a rare autoimmune disease that causes inflammation of the muscles and skin. It is most common in women over 40 years old, but it can occur in men and younger women as well. The cause of dermatomyositis is unknown, but it may be triggered by a virus or other infection. There is no cure for dermatomyositis, but treatment can help manage symptoms and improve quality of life.

There are two main types of dermatomyositis: classic and juvenile. Classic dermatomyositis is more common and usually affects adults, while juvenile dermatomyositis is less common and usually affects children. The two types share many similarities, but there are also some key differences. For example, juvenile dermatomyositis often affects the muscles of the face and neck, while classic dermatomyositis often affects the muscles of the arms and legs.

There are several different typ

In [ ]:
sample_inputs = tokenizer(
    [inference_prompt.format(sample_question) + tokenizer.eos_token],
    return_tensors="pt"
).to("cuda")

sample_outputs = model.generate(
    input_ids=sample_inputs.input_ids,
    attention_mask=sample_inputs.attention_mask,
    max_new_tokens=5000,
    eos_token_id=tokenizer.eos_token_id,
    use_cache=True,
)
sample_response = tokenizer.batch_decode(sample_outputs, skip_special_tokens=True)
print(sample_response[0].split("### Final Response:")[1])



://www.journalofethics.ama-assn.org/article/appendix-versus-ovarian-cyst-ovarian-cystadenoma/2012-08-01
ovarian cystadenoma

### Explanation:
Ovarian cystadenoma is a benign tumor that typically grows on the surface of the ovary and is filled with clear fluid. It is a benign tumor that typically grows on the surface of the ovary and is filled with clear fluid. The most common type is a serous cystadenoma, which is filled with a watery fluid. A mucinous cystadenoma is filled with a thick, sticky fluid. These are the most common types of ovarian cysts, but there are other types as well.
The most common symptoms of ovarian cystadenoma are abdominal pain, bloating, and a feeling of fullness. These symptoms may be accompanied by other symptoms, such as nausea, vomiting, and diarrhea. These symptoms may be accompanied by other symptoms, such as nausea, vomiting, and diarrhea.
The treatment for ovarian cystadenoma is usually surgery. The surgery may be laparoscopic or open. The goal of surg

### Model Training

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)


from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,

        num_train_epochs = 1, # Set this for 1 full training run.
         max_steps = 60, # Can change to 180, takes about 1 hour
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Already have LoRA adapters! We shall skip this step.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/19704 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 19,704 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.247400
2,1.099100
3,1.110100
4,1.155500
5,1.138700
6,1.102100
7,0.994600
8,1.084500
9,1.010000
10,1.181000


### Fine tuned Model Inference

In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

In [ ]:
#!zip -r lora_model.zip lora_model

In [ ]:
# !unzip lora_model.zip -d lora_model


In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
question = dataset[10]['Question']
response = dataset[10]['Response']

print("Question: ", question)
print("Response: ", response)


Question:  In a patient with dermatomyositis as indicated by fatigue, muscle weakness, a scaly rash, elevated creatine kinase-MB, anti-Jo-1 antibodies, and perimysial inflammation, which type of cancer is most often associated with this condition?
Response:  In patients with dermatomyositis, ovarian cancer is the most often associated malignancy, particularly in women. While other types of cancer such as breast, lung, and gastric can also be associated with dermatomyositis, ovarian cancer has a relatively strong and specific link, especially in the context of developing this autoimmune condition. It’s important for individuals with dermatomyositis to undergo appropriate cancer screening for early detection and management, particularly for ovarian cancer.


In [ ]:
FastLanguageModel.for_inference(model)

inputs = tokenizer(
    [inference_prompt.format(question) + tokenizer.eos_token],
    return_tensors="pt"
).to("cuda")


outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=max_seq_length,
    eos_token_id=tokenizer.eos_token_id,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print(response[0].split("### Final Response:")[1])






	"Jo-1" is an antibody that is commonly associated with dermatomyositis, particularly the anti-aminoacyl-tRNA synthetase antibody group. In this context, the presence of anti-Jo-1 antibodies is indicative of an autoimmune process involving muscle tissue. This condition is known as anti-synthetase syndrome.

Now, let's consider the question of which type of cancer is most often associated with dermatomyositis. There is a well-documented association between dermatomyositis and certain types of cancer. This association is known as the "Cancer Paraneoplastic Syndrome." While several types of cancer have been linked to dermatomyositis, the most frequently associated malignancy is ovarian cancer.

Ovarian cancer is not the only cancer associated with dermatomyositis, but it is the most common among the list of possibilities. Other types of cancer that have been linked to dermatomyositis include lung cancer, lung adenocarcinoma, and lung squamous cell carcinoma. However, ovarian cancer sta

### Chatbot

In [ ]:
%%writefile app.py

import streamlit as st
import torch
from unsloth import FastLanguageModel
from collections import deque
from transformers import BitsAndBytesConfig
# Configuration
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Initialize session state for conversation history
if "messages" not in st.session_state:
    st.session_state.messages = deque(maxlen=10)  # Buffer memory of last 10 messages
if "model" not in st.session_state:
    st.session_state.model = None
if "tokenizer" not in st.session_state:
    st.session_state.tokenizer = None

@st.cache_resource
def load_model():
    """Load the fine-tuned model and tokenizer"""
    try:
      bnb_config = BitsAndBytesConfig(
          load_in_4bit=True,
          llm_int8_enable_fp32_cpu_offload=True  # <-- offload heavy layers to CPU
      )


      model, tokenizer = FastLanguageModel.from_pretrained(
            model_name="/content/lora_model/lora_model", # change path as per your directory
            max_seq_length=max_seq_length,
            dtype=dtype,
            load_in_4bit=load_in_4bit,
            device_map="auto"
          )

      FastLanguageModel.for_inference(model)
      return model, tokenizer
    except Exception as e:
        st.error(f"Error loading model: {str(e)}")
        return None, None

def format_conversation_history():
    """Format conversation history for context"""
    history = ""
    for msg in list(st.session_state.messages)[-6:]:  # Last 6 messages for context
        role = "User" if msg["role"] == "user" else "Assistant"
        history += f"{role}: {msg['content']}\n"
    return history

def generate_response(user_input, model, tokenizer):
    """Generate response using the fine-tuned model"""

    # Get conversation context
    context = format_conversation_history()

    # Create prompt with medical expert format
    prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Answer the following medical question with clarity, accuracy, and patient safety in mind.

### Previous Conversation:
{context}

### Question:
{user_input}

### Response:
"""

    # Tokenize input
    inputs = tokenizer(prompt +tokenizer.eos_token, return_tensors="pt").to("cuda")

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
            max_new_tokens=2048,
            temperature=0.7,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the generated part (after "### Response:")
    if "### Response:" in response:
        response = response.split("### Response:")[-1].strip()

    return response

# Streamlit UI
st.title("🩺 Medical AI Chatbot")
st.markdown("*Powered by Fine-tuned Llama Model*")

# Load model if not already loaded
if st.session_state.model is None or st.session_state.tokenizer is None:
    with st.spinner("Loading model..."):
        model, tokenizer = load_model()
        if model and tokenizer:
            st.session_state.model = model
            st.session_state.tokenizer = tokenizer
            st.success("Model loaded successfully!")
        else:
            st.error("Failed to load model. Please check your model path.")
            st.stop()

# Display conversation history
st.subheader("Conversation")

# Create a container for messages
message_container = st.container()

with message_container:
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.write(message["content"])

# Chat input
if user_input := st.chat_input("Ask your medical question..."):

    # Add user message to history
    st.session_state.messages.append({"role": "user", "content": user_input})

    # Display user message
    with st.chat_message("user"):
        st.write(user_input)

    # Generate and display assistant response
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            try:
                response = generate_response(
                    user_input,
                    st.session_state.model,
                    st.session_state.tokenizer
                )
                st.write(response)

                # Add assistant response to history
                st.session_state.messages.append({"role": "assistant", "content": response})

            except Exception as e:
                error_msg = f"Error generating response: {str(e)}"
                st.error(error_msg)
                st.session_state.messages.append({"role": "assistant", "content": error_msg})

# Sidebar with information
with st.sidebar:
    st.header("⚠️ Disclaimer")
    st.warning("""
    This AI chatbot is for demonstration purposes only.
    Always consult with qualified healthcare professionals
    for medical advice and treatment decisions.
    """)

    # Clear conversation button
    if st.button("🗑️ Clear Conversation"):
        st.session_state.messages.clear()
        st.rerun()



Overwriting app.py


In [ ]:
from pyngrok import ngrok

!ngrok authtoken enter-key

# Kill previous tunnels if any
ngrok.kill()

# Set up tunnel to streamlit (default port 8501)
public_url = ngrok.connect(8501)
print("Access the app here:", public_url)


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Access the app here: NgrokTunnel: "https://c6d9ae31b33f.ngrok-free.app" -> "http://localhost:8501"


In [7]:
!streamlit run app.py --server.port 8501 &>/dev/null&
